# **Четыре карты на одном входе: внимание, attention × gradient, rollout и Chefer**

Практика к модулю [«Attention-based объяснимость»](https://ai-interpretability.school/learn/14/0/0).

В модуле разобраны четыре способа построить карту важности по вниманию трансформера: сырая карта
внимания одного слоя, attention × gradient, attention rollout и метод Chefer et al. Здесь все четыре
считаются на **одном и том же входе** — снимке с кошкой и собакой — одной и той же моделью ViT-Small,
и сравниваются числом:

1. зависит ли карта от того, про какой класс спросили;
2. насколько карты разных методов похожи друг на друга;
3. какая доля массы карты попадает в рамку кошки и в рамку собаки.

В конце — attention sink в GPT-2 small: куда уходит основная масса внимания языковой модели, и Attention Flow: та же ViT-Small, но карта считается как максимальный поток по графу внимания и сравнивается с rollout.

Вход и модель те же, что в тетради [«Атрибуция для трансформеров»](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_transformer_attribution.ipynb)
модуля «Объяснения от аксиом»: там rollout и Chefer разобраны подробно, здесь к ним добавлены сырая
карта внимания и attention × gradient. Тетрадь выполняется около двух минут на процессоре.

Автор фото кошки и мопса: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), лицензия Unsplash.

In [ ]:
!pip install timm -q     # в Colab timm бывает не установлен

In [ ]:
import io
import types
import urllib.request

import timm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = timm.create_model('vit_small_patch16_224', pretrained=True).eval()
for p in model.parameters():
    p.requires_grad_(False)


def attn_forward(self, x, **kw):
    """Внимание с сохранением матрицы и градиента по ней: обе нужны картам."""
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    q, k = self.q_norm(q), self.k_norm(k)
    attn = ((q * self.scale) @ k.transpose(-2, -1)).softmax(dim=-1)
    self.attn_map = attn          # матрица внимания нужна всем четырем картам
    attn.retain_grad()            # градиент по ней — attention × gradient и Chefer
    return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


for block in model.blocks:
    block.attn.fused_attn = False     # быстрое внимание матрицу не отдает, поэтому выключаем
    block.attn.forward = types.MethodType(attn_forward, block.attn)

cfg = model.pretrained_cfg
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(cfg['mean'], cfg['std'])])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB'))
img = img.unsqueeze(0).requires_grad_(True)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]
GRID = int(model.patch_embed.num_patches ** 0.5)
print(f'блоков: {len(model.blocks)}, решетка патчей: {GRID}x{GRID}')

## 1. Внимание и градиент по нему

Библиотека считает внимание быстрым ядром, которое саму матрицу наружу не отдает, поэтому быстрое
внимание выключено и подставлена обычная реализация. `attn.retain_grad()` сохраняет градиент логита по
матрице внимания: без него attention × gradient и Chefer посчитать нечем.

Один прогон `run(cls)` отдает матрицы внимания всех 12 блоков и градиенты логита класса `cls` по ним.

In [ ]:
def run(cls):
    """Один прогон: матрицы внимания всех блоков и градиенты логита класса cls по ним."""
    model.zero_grad()
    img.grad = None
    out = model(img)
    out[0, cls].backward()
    return [b.attn.attn_map.detach()[0] for b in model.blocks], \
        [b.attn.attn_map.grad[0] for b in model.blocks]


CAT, DOG = 281, 243        # tabby и bull mastiff — кошка и собака
mats, grads = run(DOG)
print(f'класс собаки: {names[DOG]}')
print(f'форма матрицы внимания (головы, токены, токены): {tuple(mats[0].shape)}')

## 2. Четыре метода

Все четыре берут строку токена `CLS` — то, сколько он забирает у каждого патча, — и раскладывают ее
обратно в решетку патчей $14\times14$:

- **attention** — сырая карта: внимание последнего блока, усредненное по головам.
- **attention × gradient** — та же матрица, поэлементно умноженная на градиент логита класса по ней;
  берется положительная часть и среднее по головам. Это ответ на главную претензию Jain и Wallace:
  вес внимания сам по себе не говорит, повлиял ли он на ответ, а градиент говорит.
- **rollout** — внимание всех блоков, усредненное по головам, с добавленной единичной матрицей за
  обходное соединение, перемноженное по слоям.
- **Chefer** — generic-вариант из работы [Chefer et al. (2021b)](https://arxiv.org/abs/2103.15679): каждая матрица перед
  усреднением умножена на градиент логита по ней, и по слоям накапливается $R \leftarrow R + \bar A R$
  без нормировки строк, как в коде авторов. Полный вариант из статьи добавляет правила LRP для
  обходных соединений и произведения $A\cdot V$; классовую специфичность дает градиентный множитель,
  и он здесь есть.

In [ ]:
def raw(mats, grads):
    """Сырая карта: внимание последнего блока, среднее по головам."""
    return mats[-1].mean(0)


def attn_x_grad(mats, grads):
    """Attention × gradient: внимание последнего блока, умноженное на градиент логита, положительная часть, среднее по головам."""
    return (grads[-1] * mats[-1]).clamp(min=0).mean(0)


def rollout(mats, grads):
    """Attention rollout: среднее по головам, половина веса на обходное соединение, произведение по слоям."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A in mats:
        r = (0.5 * A.mean(0) + 0.5 * torch.eye(n)) @ r
    return r


def chefer(mats, grads):
    """Chefer et al. (generic): та же прокрутка, но каждая матрица взвешена градиентом логита класса."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A, G in zip(mats, grads):
        a = (G * A).clamp(min=0).mean(0) + torch.eye(n)   # R <- R + A·R, как в коде авторов: строки не нормируются
        r = a @ r
    return r


METHODS = {'attention': raw, 'attention x gradient': attn_x_grad, 'rollout': rollout, 'Chefer': chefer}


def to_map(r):
    """Строка токена CLS без него самого, разложенная обратно в решетку патчей и нормированная в [0, 1]."""
    m = r[0, 1:].reshape(GRID, GRID)
    return (m - m.min()) / (m.max() - m.min() + 1e-9)


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-9))

## 3. Зависит ли карта от вопроса

Строим по две карты каждым методом — спрашивая про кошку и про собаку — и считаем корреляцию Пирсона
между ними. Сначала для двух методов без градиента.

In [ ]:
maps = {}
for cls, label in ((CAT, 'кошка'), (DOG, 'собака')):
    m, g = run(cls)
    maps[label] = {name: to_map(f(m, g)) for name, f in METHODS.items()}

for name in ('attention', 'rollout'):
    a, b = maps['кошка'][name], maps['собака'][name]
    print(f'{name:22} корреляция карт кошки и собаки: {corr(a, b):.6f}   максимальное расхождение: {(a - b).abs().max():.2e}')

**У сырой карты внимания и у rollout корреляция ровно единица, а расхождение ровно ноль.** Матрица
внимания считается на прямом проходе и от класса не зависит, поэтому ни она, ни произведение
матриц от вопроса не зависят.

**Задание 1.** Посчитайте то же самое для attention × gradient и для Chefer. Какая из двух карт
сильнее меняется, когда вопрос меняется с собаки на кошку? Ответ — корреляция карт attention × gradient
для кошки и собаки, до сотых.

In [ ]:
# Ваш код здесь

## 4. Похожи ли карты разных методов

Корреляция Пирсона между картами четырех методов на одном и том же вопросе — про собаку.

In [ ]:
# корреляция карт четырех методов, вопрос про собаку
print(f'{"":22}' + ''.join(f'{n:>22}' for n in METHODS))
for a in METHODS:
    print(f'{a:22}' + ''.join(f'{corr(maps["собака"][a], maps["собака"][b]):22.3f}' for b in METHODS))

Таблица симметрична, на диагонали единицы. Найдите в строке `Chefer` метод, чья карта к ней
ближе всего, и метод, чья карта дальше всего.

## 5. Куда именно смотрят карты

Корреляция говорит, что карты разные, но не говорит, какая из них смотрит туда, куда надо. Разметим
рамки кошки и собаки и посмотрим, какая доля массы каждой карты попадает в каждую рамку.

In [ ]:
# Рамки размечены на глаз по снимку 224 на 224, те же, что в тетради «Атрибуция для трансформеров»
BOXES = {'кошка': (10, 40, 110, 200), 'собака': (115, 30, 215, 205)}


def mass_in(m, box):
    """Доля массы карты, попавшая внутрь рамки."""
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


print(f'{"":22}{"спросили":10}{"в рамке кошки":>16}{"в рамке собаки":>17}')
for label in ('кошка', 'собака'):
    for name in METHODS:
        m = maps[label][name]
        print(f'{name:22}{label:10}{mass_in(m, BOXES["кошка"]) * 100:15.1f}%'
              f'{mass_in(m, BOXES["собака"]) * 100:16.1f}%')

**Строки сырой карты внимания и rollout для кошки и для собаки совпадают** — эти карты отвечают на
вопрос «куда вообще смотрит модель». **Строки attention × gradient и Chefer расходятся**: спросили про
кошку — больше массы легло в рамку кошки.

**Задание 2.** Сырая карта внимания кладет в рамку собаки больше массы, чем rollout, и при вопросе
про кошку тоже. Значит ли это, что модель «смотрит на собаку», когда решает, есть ли на снимке кошка?
Опирайтесь на числа таблицы.

## 6. Посмотреть глазами

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, label in enumerate(('кошка', 'собака')):
    for col, name in enumerate(METHODS):
        axes[row, col].imshow(maps[label][name].numpy(), cmap='inferno')
        axes[row, col].set_title(f'{name}: {label}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 7. Attention sink: куда уходит масса внимания в языковой модели

[Xiao et al. (2024)](https://arxiv.org/abs/2309.17453) заметили, что в обученных авторегрессионных
языковых моделях большая доля внимания уходит в первый токен последовательности, какой бы смысл у
него ни был. Softmax обязан распределить единицу массы, и модели выгодно складывать «лишнее» внимание
туда, где оно ничего не испортит. Проверим на GPT-2 small: для каждого слоя посчитаем, какую долю
внимания токены с позиции 1 и дальше отдают первому токену, в среднем по 12 головам и по позициям. Для
сравнения — та же доля при равномерном внимании, когда токен на позиции $i$ делит массу на $i+1$ токенов.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained('gpt2')
lm = AutoModelForCausalLM.from_pretrained('gpt2', attn_implementation='eager').eval()

text = 'When Mary and John went to the store, John gave a drink to Mary.'
ids = tok(text, return_tensors='pt')
with torch.no_grad():
    att = lm(**ids, output_attentions=True).attentions      # 12 тензоров (батч, головы, токены, токены)

n = ids['input_ids'].shape[1]
shares = torch.stack([a[0, :, 1:, 0].mean() for a in att])  # доля первого токена, средняя по головам и позициям 1..n-1
uniform = float((1.0 / torch.arange(2, n + 1, dtype=torch.float32)).mean())
print(f'токенов: {n}, первый токен: {tok.convert_ids_to_tokens(ids["input_ids"][0, :1])}')
for layer, s in enumerate(shares, start=1):
    print(f'слой {layer:2}: {float(s):.3f}')
print(f'доля при равномерном внимании: {uniform:.3f}')

**Задание 3.** В каком слое доля внимания на первом токене максимальна? Возьмите карту внимания
этого слоя, усредненную по головам, вычеркните первый токен (строку и столбец), перенормируйте строки
и найдите, на какой токен после этого больше всего смотрит последний токен фразы.

In [ ]:
# Ваш код здесь

## 8. Attention Flow: максимальный поток вместо произведения

Rollout и Attention Flow ([Abnar и Zuidema, 2020](https://arxiv.org/abs/2005.00928)) читают один и тот же граф. Вершины — токены по слоям: $(l, i)$, где $l=0$ — вход первого блока, $l=12$ — выход последнего. Ребро из $(l-1, j)$ в $(l, i)$ получает емкость

$$\tilde A^{(l)}_{ij}=\tfrac12\,\mathbb E_h A^{(l)}_{ij}+\tfrac12\,\delta_{ij},$$

то есть внимание блока $l$, усредненное по головам, с половиной веса на обходное соединение, как в функции `rollout` из раздела 2. Rollout складывает по всем путям из входного патча $j$ в токен `CLS` последнего слоя произведения емкостей вдоль пути. Attention Flow решает задачу о максимальном потоке из вершины $(0, j)$ в вершину $(12, \texttt{CLS})$: сколько можно пропустить по графу, не превысив емкость ни одного ребра. Максимальный поток равен емкости минимального разреза, поэтому его задает самое узкое место графа, а не произведение весов.

Задача о потоке решается для каждого входного патча отдельно. На снимке $224\times224$ это 196 задач на графе с $12\cdot197^2\approx466$ тысяч ребер, и одна карта считается минутами. Поэтому здесь вход короткий: тот же снимок, сжатый до $112\times112$, решетка $7\times7$, 50 токенов и около 30 тысяч ребер. ViT-Small принимает такой вход, если создать его с `dynamic_img_size=True`: позиционные эмбеддинги интерполируются под новую решетку. Rollout для сравнения считается на том же коротком входе.

In [ ]:
import time
import networkx as nx

model112 = timm.create_model('vit_small_patch16_224', pretrained=True, dynamic_img_size=True).eval()
for p in model112.parameters():
    p.requires_grad_(False)
for block in model112.blocks:
    block.attn.fused_attn = False
    block.attn.forward = types.MethodType(attn_forward, block.attn)

tf112 = transforms.Compose([transforms.Resize((112, 112)), transforms.ToTensor(),
                            transforms.Normalize(cfg['mean'], cfg['std'])])
img112 = tf112(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB'))
img112 = img112.unsqueeze(0).requires_grad_(True)   # attn_forward сохраняет градиент, ему нужен вход с градиентом
model112(img112)                                     # только прямой проход: нужны матрицы внимания
mats112 = [b.attn.attn_map.detach()[0] for b in model112.blocks]
N112 = mats112[0].shape[-1]
GRID112 = int((N112 - 1) ** 0.5)
print(f'токенов: {N112}, решетка патчей: {GRID112}x{GRID112}')


def capacities(mats):
    # емкости ребер: среднее по головам и половина веса на обходное соединение, как в rollout
    n = mats[0].shape[-1]
    return [0.5 * A.mean(0) + 0.5 * torch.eye(n) for A in mats]


def attention_flow(mats):
    # максимальный поток из каждого входного патча в токен CLS последнего слоя
    caps = capacities(mats)
    n = caps[0].shape[0]
    G = nx.DiGraph()
    for l, A in enumerate(caps):
        for i in range(n):
            for j in range(n):
                G.add_edge((l, j), (l + 1, i), capacity=float(A[i, j]))
    return torch.tensor([nx.maximum_flow_value(G, (0, j), (len(caps), 0)) for j in range(1, n)])


def rollout_cls(mats):
    # строка токена CLS у rollout из раздела 2, без самого CLS
    return rollout(mats, None)[0, 1:]


# проверка на одном блоке: путь из патча в CLS один — прямое ребро, и поток обязан совпасть с rollout
one_flow, one_roll = attention_flow(mats112[-1:]), rollout_cls(mats112[-1:])
print(f'один блок: корреляция потока и rollout {corr(one_flow, one_roll):.6f}, '
      f'максимальное расхождение {(one_flow - one_roll).abs().max():.1e}')

На одном блоке поток и rollout совпадают до ошибки округления: при единственном пути поток равен емкости его единственного ребра, а произведение из одного множителя — самому множителю. Теперь все 12 блоков.

In [ ]:
start = time.time()
flow12 = attention_flow(mats112)
roll12 = rollout_cls(mats112)
print(f'12 блоков: {N112 - 1} задач о потоке за {time.time() - start:.1f} с')
print(f'корреляция карт Attention Flow и rollout: {corr(flow12, roll12):.3f}')
print(f'поток патча: от {float(flow12.min()):.3f} до {float(flow12.max()):.3f}, '
      f'патчей с потоком больше 0.999: {int((flow12 > 0.999).sum())} из {N112 - 1}')

Карта потока с картой rollout связана слабо, и у части патчей поток равен единице. Найдем, где проходит минимальный разрез. Из вершины $(0, j)$ выходят ребра в первый блок с общей емкостью $\sum_i\tilde A^{(1)}_{ij}$, сумма по столбцу: столько первый блок забирает у патча $j$. В вершину $(12, \texttt{CLS})$ входит не больше единицы: строка внимания нормирована. Обходное соединение с весом 0.5 держит у каждого токена собственный путь через все слои, и путей между краями графа много. Если разрез проходит по краям, поток патча равен $\min\big(\sum_i\tilde A^{(1)}_{ij},\,1\big)$. Проверим это прямо.

In [ ]:
first_block = capacities(mats112)[0].sum(0)[1:].clamp(max=1)   # сколько первый блок забирает у патча, не больше единицы
print(f'корреляция потока и емкости выхода из патча в первый блок: {corr(flow12, first_block):.6f}, '
      f'максимальное расхождение {(flow12 - first_block).abs().max():.1e}')


def to_map112(v):
    # карта по патчам короткого входа, нормированная в [0, 1]
    m = v.reshape(GRID112, GRID112)
    return (m - m.min()) / (m.max() - m.min() + 1e-9)


flow_maps = {'rollout': to_map112(roll12), 'Attention Flow': to_map112(flow12),
             'первый блок': to_map112(first_block)}
print(f'\n{"":16}{"в рамке кошки":>16}{"в рамке собаки":>17}')
for name, m in flow_maps.items():
    print(f'{name:16}{mass_in(m, BOXES["кошка"]) * 100:15.1f}%{mass_in(m, BOXES["собака"]) * 100:16.1f}%')

Корреляция единица, расхождение на уровне ошибки округления: на этом входе Attention Flow — это карта того, сколько внимания первый блок отдает каждому патчу, ограниченная сверху единицей. Остальные одиннадцать блоков в поток не попадают, потому что ни одно ребро внутри графа не уже краев. Rollout, наоборот, перемножает все двенадцать блоков, поэтому доли массы в рамках у двух карт разные.

**Задание 4.** Узкое место сдвигается, если граф короче. Постройте Attention Flow и rollout только по трем последним блокам (`mats112[-3:]`) и посчитайте корреляцию Пирсона двух карт. Ответ — до сотых.

In [ ]:
# Ваш код здесь

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8))
for ax, (name, m) in zip(axes, flow_maps.items()):
    ax.imshow(m.numpy(), cmap='inferno')
    ax.set_title(f'{name}, 12 блоков' if name != 'первый блок' else 'емкость выхода в первый блок')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Что унести из тетради

- **Сырая карта внимания и rollout классово слепы**, и это проверяется побитово: корреляция карт для
  двух классов равна единице. Они показывают, куда течет информация, а не что повлияло на ответ.
- **Класс попадает в карту только через градиент.** Attention × gradient и Chefer отличаются от
  своих слепых двойников одним множителем — градиентом логита по матрице внимания.
- **Карты разных методов расходятся** даже на одном входе и одной модели. Выбирать метод надо по
  вопросу, а не по красоте карты.
- **В языковых моделях первый токен забирает основную массу внимания.** Прежде чем читать карту
  внимания, этот столбец надо вычесть.
- **Attention Flow задается самым узким местом графа, а на ViT-Small с обходными соединениями оно у входа.** Поток патча равен тому, сколько внимания отдает ему первый блок, ограниченному единицей, и с rollout такая карта связана слабо. Задача о потоке решается для каждого входного токена, поэтому вход пришлось сжать до 50 токенов.